In [1]:
import sys
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

root = str(Path.cwd().parent)

if root not in sys.path:
    sys.path.append(root)

from src.step2_transformation import DataTransformer
from src.step3a_bwm_model import BWMCalculator
from src.step3b_hierarchical_bwm import HierarchicalAggregator
from src.step4_saw_aggregation import SAWCalculator

In [2]:
# ==========================================
# 2. SIMULATION: USER PROFILE
# ==========================================
# Instantiate the aggregator (it will automatically read step1_features_config.yaml)
aggregator = HierarchicalAggregator(config_name='step1_features_config.yaml')

# LAYER 1 (MACRO): Imagine BWM calculated these weights for the 5 main areas
macro_weights = {
    'cost': 0.45,        # Cost drives the decision (45%)
    'safety': 0.30,      # Safety comes second (30%)
    'performance': 0.15, # Engine/Performance (15%)
    'comfort': 0.07,     # Comfort (7%)
    'aesthetics': 0.03   # Aesthetics barely matter (3%)
}

✅ Hierarchical Groups loaded: ['cost', 'safety', 'performance', 'comfort', 'aesthetics']


In [3]:
# LAYER 2 (MICRO / SPECIFICS):
# The user opened the "Advanced" tab ONLY in the Safety category.
# They stated: "I insist on a Rear View Camera and Airbags, I don't care much about the rest".
# For the other categories, they left it blank, so the system will divide them equally.
custom_micro_weights = {
    'safety': {
        'rear_view_camera': 0.50,      # Half of the safety score goes to the Camera
        'airbag': 0.40,                # 40% goes to Airbags
        'rear_parking_sensors': 0.05,
        'tire_pressure_sensor': 0.05
    }
}

In [4]:
# ==========================================
# 3. HIERARCHICAL MULTIPLICATION EXECUTION
# ==========================================
absolute_weights = aggregator.generate_flat_weights(macro_weights, custom_micro_weights)

📊 Absolute Weights generated for 28 features (Sum: 0.9996)


In [5]:
# ==========================================
# 4. VISUALIZATION: THE REAL WEIGHT OF EACH ITEM
# ==========================================
# Transform into a DataFrame for elegant visualization
df_final_weights = pd.DataFrame(list(absolute_weights.items()), columns=['Feature', 'Absolute Weight in Final Score'])
df_final_weights = df_final_weights.sort_values(by='Absolute Weight in Final Score', ascending=False).reset_index(drop=True)

print("🏆 ABSOLUTE IMPORTANCE RANKING (Top 10 Features):")
print("How your engine translated the macro categories into the mathematical matrix:")

# Display the table with visual gradient formatting (Purples to differentiate from the previous ones)
display(df_final_weights)

🏆 ABSOLUTE IMPORTANCE RANKING (Top 10 Features):
How your engine translated the macro categories into the mathematical matrix:


,Feature,Absolute Weight in Final Score
0,rear_view_camera,0.1500
1,airbag,0.1200
2,cost,0.1125
3,warranty,0.1125
4,highway_fuel_economy,0.1125
5,city_fuel_economy,0.1125
6,horsepower,0.0300
7,engine,0.0300
8,torque,0.0300
9,ground_clearance,0.0300
